In [1]:
import pandas as pd
import sqlite3

## Connect to the SQLite database
conn = sqlite3.connect("supply_chain.db")


# 7. Advanced SQL

## Objective

This section demonstrates a few advanced SQL concepts commonly used in business analytics and technical interviews.

Topics Covered

- Common Table Expressions (CTEs)
- ROW_NUMBER()
- RANK()

In [2]:
## Common Table Expressions (CTEs)
## Which products generated revenue above the average product revenue?
pd.read_sql("""
WITH ProductRevenue AS
(
SELECT

p.product_name,

SUM(f.delivery_qty * p.price_INR) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id = p.product_id

GROUP BY p.product_name
)

SELECT *

FROM ProductRevenue

WHERE revenue >
(
SELECT AVG(revenue)

FROM ProductRevenue
)

ORDER BY revenue DESC;
""", conn)


,product_name,revenue
0,AM Butter 500,121907700
1,AM Biscuits 750,69372900
2,AM Milk 500,64393375
3,AM Butter 250,57480000
4,AM Biscuits 500,45719200
5,AM Tea 500,37137375
6,AM Milk 250,32985612


### Business Insight

Products generating revenue above the average should receive higher inventory priority and marketing focus because they contribute significantly to overall business performance.

In [3]:
## ROW_NUMBER()
## Assign a unique ranking to all products based on revenue.
pd.read_sql("""
SELECT

product_name,

revenue,

ROW_NUMBER() OVER(
ORDER BY revenue DESC
) AS row_number

FROM
(
SELECT

p.product_name,

SUM(f.delivery_qty*p.price_INR) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id=p.product_id

GROUP BY p.product_name
);
""", conn)


,product_name,revenue,row_number
0,AM Butter 500,121907700,1
1,AM Biscuits 750,69372900,2
2,AM Milk 500,64393375,3
3,AM Butter 250,57480000,4
4,AM Biscuits 500,45719200,5
5,AM Tea 500,37137375,6
6,AM Milk 250,32985612,7
7,AM Biscuits 250,23590700,8
8,AM Butter 100,22841940,9
9,AM Curd 250,22233250,10


### Business Insight

ROW_NUMBER() assigns a unique sequential number to every product, even when two products have the same revenue.

It is useful when selecting the Top N records.

In [4]:
## RANK()
## Rank products according to their revenue
pd.read_sql("""
SELECT

product_name,

revenue,

RANK() OVER(
ORDER BY revenue DESC
) AS revenue_rank

FROM
(
SELECT

p.product_name,

SUM(f.delivery_qty*p.price_INR) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id=p.product_id

GROUP BY p.product_name
);
""", conn)

,product_name,revenue,revenue_rank
0,AM Butter 500,121907700,1
1,AM Biscuits 750,69372900,2
2,AM Milk 500,64393375,3
3,AM Butter 250,57480000,4
4,AM Biscuits 500,45719200,5
5,AM Tea 500,37137375,6
6,AM Milk 250,32985612,7
7,AM Biscuits 250,23590700,8
8,AM Butter 100,22841940,9
9,AM Curd 250,22233250,10


### Business Insight

RANK() is useful for comparing product performance. If two products have the same revenue, they receive the same rank, and the next rank is skipped.

### Business Question

Which products are the highest revenue generators within each product category?

In [5]:
pd.read_sql("""
SELECT

category,

product_name,

revenue,

DENSE_RANK() OVER(
PARTITION BY category
ORDER BY revenue DESC
) AS category_rank

FROM
(
SELECT

p.category,

p.product_name,

SUM(f.delivery_qty * p.price_INR) AS revenue

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY
p.category,
p.product_name
)

ORDER BY
category,
category_rank;
""", conn)

,category,product_name,revenue,category_rank
0,Dairy,AM Butter 500,121907700,1
1,Dairy,AM Milk 500,64393375,2
2,Dairy,AM Butter 250,57480000,3
3,Dairy,AM Milk 250,32985612,4
4,Dairy,AM Butter 100,22841940,5
5,Dairy,AM Curd 250,22233250,6
6,Dairy,AM Ghee 250,17732700,7
7,Dairy,AM Milk 100,12547550,8
8,Dairy,AM Ghee 150,10337625,9
9,Dairy,AM Curd 100,8985100,10


### Business Insights

- **AM Butter 500** is the highest revenue-generating product in the **Dairy** category, making it the category's flagship product.

- **AM Biscuits 750** ranks first in the **Food** category, indicating strong customer demand for larger pack sizes.

- **AM Tea 500** is the top-performing product in the **Beverages** category, contributing the highest revenue within its category.

- The Dairy category has **12 products**, providing a broad product portfolio, whereas the Food and Beverages categories have fewer products. This suggests that Dairy contributes significantly to overall business revenue.

- Ranking products within each category helps identify the best-performing SKUs for inventory planning, procurement, pricing strategies, and promotional campaigns.

- Lower-ranked products such as **AM Curd 50**, **AM Ghee 100**, and **AM Tea 100** generate comparatively lower revenue and may require promotional activities or portfolio review to improve their performance.

Why did you use DENSE_RANK() instead of RANK()?

"I wanted to rank products within each category. DENSE_RANK() assigns the same rank to products with equal revenue without skipping the next rank. This makes business reports easier to interpret, especially when comparing product performance within categories."